# Simple Resale Scraper (No API) — Size Distribution

This notebook demonstrates a **minimal, polite** scraper for a public resale website (example: **Depop**) to pull listings for a query (e.g., *Levi’s 501 women jeans*), extract size info from the **description and title**, and plot a size distribution.

⚠️ **Important**
- Check the site’s **robots.txt** and **Terms of Service** before scraping. Do not bypass CAPTCHAs, logins, or paywalls.
- Scrape **slowly** and avoid heavy concurrency; this notebook uses small delays.
- Results and selectors may need tweaking because marketplaces change their HTML frequently.

You can adapt the selectors for other marketplaces’ public listing pages later.

## 1) Configuration
Edit the search query and basic limits here. For a first test, keep the limits small.

In [7]:
SEARCH_QUERY = "levis 501 women jeans"   # change me to any item
MAX_LISTINGS = 80                         # cap the number of listing pages to visit
SCROLL_PASSES = 6                         # how many times to scroll the search result page
DELAY_SEC = (1.0, 2.5)                    # random delay bounds between visits

# Target site: Depop search (public)
SITE_BASE = "https://www.depop.com"
SEARCH_URL = SITE_BASE + "/search/?q="

## 2) Helpers — polite scraping, parsing, normalization
We use Playwright to render JS, BeautifulSoup to parse HTML, and simple regex rules to parse sizes.

In [8]:
import re, time, random
from urllib.parse import urljoin, quote_plus
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

ALPHA_MAP = {"xxs":"XXS","xs":"XS","s":"S","m":"M","l":"L","xl":"XL","xxl":"XXL","2xl":"XXL","3xl":"3XL"}

def parse_size_text(text: str | None):
    if not text:
        return None
    t = text.lower()
    # pattern: 28x30 or 28 x 30 or 28/30
    m = re.search(r"\b(\d{2})\s*[x×/]\s*(\d{2})\b", t)
    if m:
        return f"{m.group(1)}x{m.group(2)}"
    # 'waist 28' or 'size 28'
    m = re.search(r"\b(?:size|sz|waist)\s*[:\s]*?(\d{2})\b", t)
    if m:
        return m.group(1)
    # alpha sizes
    m = re.search(r"\b(xxs|xs|s|m|l|xl|xxl|2xl|3xl)\b", t)
    if m:
        return ALPHA_MAP.get(m.group(1), m.group(1).upper())
    return None

def extract_listing_fields(html: str, url: str):
    soup = BeautifulSoup(html, "html.parser")
    # These selectors are intentionally simple; inspect the page to refine them for better accuracy.
    title_el = soup.select_one("h1, .item-title, h2")
    price_el = soup.select_one("[data-test='price'], .price, .ProductPrice, .Money")
    desc_el  = soup.select_one(".description, [data-test='description'], .item-description, article")
    seller_el = soup.select_one(".seller, [data-test='seller'], .user-card, a[href*='/@']")
    # Fallbacks
    title = title_el.get_text(strip=True) if title_el else ""
    price = price_el.get_text(strip=True) if price_el else ""
    desc  = desc_el.get_text("\n", strip=True) if desc_el else ""
    seller= seller_el.get_text(strip=True) if seller_el else ""
    size = parse_size_text(desc) or parse_size_text(title)
    return {
        "url": url,
        "title": title,
        "price": price,
        "description": desc,
        "seller": seller,
        "size_parsed": size,
    }

def polite_sleep(bounds=(1.0,2.5)):
    time.sleep(random.uniform(*bounds))

## 3) Crawl search results → visit listings
This will:
1. Load the search results page
2. Scroll a few times to load more items
3. Collect candidate listing links
4. Visit each listing and extract fields (title, price, description, size)

Keep the caps low at first to stay polite and avoid getting blocked.

In [9]:
search_url = SEARCH_URL + quote_plus(SEARCH_QUERY)
records = []
visited = set()

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    context = browser.new_context(user_agent=(
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/118 Safari/537.36"
    ))
    page = context.new_page()
    print("Loading:", search_url)
    page.goto(search_url, timeout=60000)
    polite_sleep(DELAY_SEC)
    # Scroll to load more results
    for _ in range(SCROLL_PASSES):
        page.evaluate("window.scrollBy(0, document.body.scrollHeight)")
        polite_sleep((0.7, 1.5))

    html = page.content()
    soup = BeautifulSoup(html, "html.parser")
    anchors = soup.select("a")
    candidate_links = []
    for a in anchors:
        href = a.get("href")
        if not href:
            continue
        # crude filter: item/detail-like paths; refine as you learn the site structure
        if "/listing/" in href or "/items/" in href or href.count("/") >= 3:
            full = urljoin(SITE_BASE, href)
            candidate_links.append(full)

    # de-duplicate and cap
    uniq_links = []
    for u in candidate_links:
        if u not in visited and u not in uniq_links and len(uniq_links) < MAX_LISTINGS:
            uniq_links.append(u)

    print(f"Found {len(uniq_links)} candidate links; visiting up to {MAX_LISTINGS}...")
    for link in tqdm(uniq_links):
        try:
            page.goto(link, timeout=45000)
            polite_sleep(DELAY_SEC)
            item_html = page.content()
            rec = extract_listing_fields(item_html, link)
            records.append(rec)
            visited.add(link)
        except Exception as e:
            print("Error visiting", link, e)
            polite_sleep((2.0, 3.0))

    browser.close()

df = pd.DataFrame(records)
print("Rows scraped:", len(df))
df.head(10)

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

## 4) Size normalization & simple distribution
Normalize parsed sizes and plot a basic distribution.

In [ ]:
def normalize_size(s):
    if not isinstance(s, str) or not s.strip():
        return None
    s0 = s.strip().upper()
    s0 = re.sub(r"\s*[X×]\s*", "x", s0)
    if re.match(r"^\d{2}x\d{2}$", s0):
        return s0
    if re.match(r"^\d{2}$", s0):
        return s0
    if s0 in {"XXS","XS","S","M","L","XL","XXL","3XL"}:
        return s0
    m = re.search(r"(SIZE|WAIST)\s*(\d{2})\b", s0)
    if m:
        return m.group(2)
    return None

dist = (df.assign(size_norm=lambda d: d["size_parsed"].map(normalize_size))
          .dropna(subset=["size_norm"])['size_norm']
          .value_counts()
          .reset_index())
dist.columns = ["size", "count"]
dist

In [ ]:
# Plot
def sort_key(x):
    m = re.match(r"^(\d{2})(?:x(\d{2}))?$", x)
    if m:
        return (0, int(m.group(1)), int(m.group(2) or 0))
    alpha_order = {"XXS":0,"XS":1,"S":2,"M":3,"L":4,"XL":5,"XXL":6,"3XL":7}
    return (1, alpha_order.get(x, 999), 0)

dist_sorted = dist.sort_values(by="size", key=lambda s: s.map(sort_key))
plt.figure(figsize=(10,5))
plt.bar(dist_sorted["size"], dist_sorted["count"])
plt.xticks(rotation=45, ha='right')
plt.title(f"Size distribution for: {SEARCH_QUERY}")
plt.xlabel("Size (normalized)")
plt.ylabel("Count of listings")
plt.tight_layout()
plt.show()

## 5) Save results
Export the raw table to CSV for later analysis.

In [ ]:
out_csv = "listings_scraped.csv"
df.to_csv(out_csv, index=False)
out_csv

### Notes & next steps
- Inspect the site’s HTML and refine the selectors for `title`, `price`, `description`, and `seller`.
- Improve the size parser with more patterns (e.g., rise/inseam, bust/hip for tops).
- Add de-duplication (e.g., by URL or image hash) if you crawl multiple pages.
- To target another site: change `SITE_BASE`, `SEARCH_URL` pattern, and link filters.
- Consider caching HTML locally while iterating, so you don’t re-fetch pages repeatedly.